# Fine-tune on your Mac's GPU: LoRA on `mps` with Flyte

A devbox task that asks for `gpu="AppleSilicon"` runs natively on this Mac, with PyTorch on the Metal GPU, while Flyte keeps the run graph, caching, logs and reports. We fine-tune Qwen2.5-0.5B-Instruct for text-to-SQL with LoRA, score it before and after, and serve the adapter.

| Step | What it shows |
|---|---|
| 0 | Where the task runs: platform, chip, `mps`, GPU vs CPU matmul. No downloads. |
| 1 | The baseline. Fine-tuning needs a gap to close. |
| 2 | The pipeline: data → baseline → LoRA → evaluate, with a live loss chart. |
| 3 | Serve the adapter as an API on the same GPU. |

See the README for results and the explanation of how a task gets onto the Mac.

---

## Setup

This notebook runs **on your Mac**, next to an Apple Silicon devbox. It is not a Colab notebook: Colab has no Metal GPU and no devbox. (The code still runs there with `flyte run --local` and `ACCELERATOR=cpu`, just slowly.)

Before the first cell:

```bash
flyte start devbox --apple-silicon
cd tutorials/apple-silicon-fine-tuning
flyte create config --devbox
uv venv .venv --python 3.12 && source .venv/bin/activate && uv pip install -r requirements.txt pygments jupyter
```

In [ ]:
import os, platform
assert platform.machine() == 'arm64' and platform.system() == 'Darwin', 'This notebook expects an Apple Silicon Mac'
from utils.file_viewer import view_file
from utils.report_viewer import run_link, show_latest_local
!flyte get devbox --no-probes | head -5

### Running the steps

| | |
|---|---|
| `flyte run ...` | On the devbox. GPU tasks run natively on this Mac; every run gets a page in the UI at http://localhost:30080 with logs and reports. |
| `flyte run --local ...` | In this process, no devbox. On a Mac the GPU tasks still use `mps`. Reports land under `/tmp/flyte/metadata`; `show_latest_local()` renders the newest one. |

The cells below use the devbox. Each prints a run URL: open it for the live report.

### 0. Prove the task is on the GPU

`gpu_check` prints `Darwin arm64`, the chip name, `mps`, and a 2048×2048 matmul on the GPU versus the CPU. Its report has the bar chart. The first run builds the Linux image (torch has to be in it) and the Mac environment; both are cached.

In [ ]:
!flyte run workflow.py gpu_check

In [ ]:
view_file("config.py", title="the ACCELERATOR knob", show_path=True)
view_file("sql.py", title="device choice and scoring", show_path=True)

### 1. The baseline

Score the untouched model on 100 held-out questions. Exact match on normalized SQL, so the number is a floor; open the **evaluate** task's report for the side-by-side.

In [ ]:
!flyte run workflow.py baseline --num_eval_examples 50

### 2. The pipeline

Data → baseline → LoRA → evaluate. Open the **train** task while it runs for the live loss chart; the pipeline's own report has the before/after bars. On a 16 GB Mac add `--batch_size 2 --grad_accum 4`.

In [ ]:
!flyte run workflow.py pipeline

In [ ]:
view_file("workflow.py", title="the tasks", show_path=True)

### Same thing, no devbox

`--local` runs in this process. `pick_device('apple')` still picks `mps`, so this is the Mac GPU with none of the plumbing (and none of the run graph).

In [ ]:
!flyte run --local workflow.py gpu_check
show_latest_local()

### 3. Serve the adapter

`serve.py` mounts the adapter from the latest pipeline run (`RunOutput`) and serves it on the same accelerator.

In [ ]:
!python serve.py

In [ ]:
import httpx
URL = "http://apple-silicon-sql-api-flytesnacks-development.localhost:30081"
print(httpx.get(f"{URL}/health", timeout=120).json())
r = httpx.post(f"{URL}/generate", timeout=120, json={
    "schema": "CREATE TABLE orders (order_id INT, customer_id INT, product VARCHAR, quantity INT, price DECIMAL)",
    "question": "Which product had the most orders?",
})
print(r.json())

In [ ]:
view_file("serve.py", title="the app", show_path=True)

### The same code on a cluster

```bash
ACCELERATOR=T4 flyte run workflow.py pipeline
ACCELERATOR=T4 python serve.py
```

Point `.flyte/config.yaml` at a cluster and the knob picks `gpu="T4:1"` and `cuda`. Nothing else changes.